In [ ]:
# --- CELL 1: INITIALIZATION & INGESTION (v3.0 - Clean Room) ---

import os
import sys

# 1. ENVIRONMENT AUDIT (The Sentinel)
# Check for rogue files that cause circular imports
rogue_files = [f for f in os.listdir('.') if f in ['pandas.py', 'io.py', 'csv.py']]
if rogue_files:
    print(f"!!! CRITICAL ALERT: ROGUE FILES DETECTED: {rogue_files} !!!")
    print("ACTION: Delete these files immediately via the file browser on the left.")
    # Stop execution to prevent the error
    sys.exit("Execution halted due to rogue files.")
else:
    print("--- ENVIRONMENT CLEAN. NO ROGUE FILES DETECTED. ---")

# 2. IMPORTS (Safe Mode)
try:
    import pandas as pd
    print(f"--- PANDAS VERSION: {pd.__version__} LOADED SUCCESSFULLY ---")
    from google.colab import drive
    from google.colab import auth
    import gspread
    from google.auth import default
except AttributeError as e:
    print("!!! ENVIRONMENT CORRUPTION DETECTED !!!")
    print("Please perform: Runtime -> Disconnect and Delete Runtime.")
    raise e

# 3. MOUNT DRIVE
if not os.path.exists('/content/drive'):
    print("--- MOUNTING DRIVE ---")
    drive.mount('/content/drive')

# 4. AUTHENTICATE
print("--- AUTHENTICATING USER ---")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# --- CONFIGURATION ---
SOURCE_SHEET_NAME = 'raw_Offers'
SOURCE_WORKSHEET_NAME = 'diamond_offers'

# PLACEHOLDER PATH - Update this!
OUTPUT_CSV_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_1/cher_ami_v1.csv'

# 5. INGEST
print(f"--- INGESTING: '{SOURCE_SHEET_NAME}' // '{SOURCE_WORKSHEET_NAME}' ---")

try:
    sh = gc.open(SOURCE_SHEET_NAME)
    worksheet = sh.worksheet(SOURCE_WORKSHEET_NAME)
    rows = worksheet.get_all_values()

    # Convert to DataFrame
    offers_df = pd.DataFrame(rows[1:], columns=rows[0])

    print(f"--- SUCCESS: Ingested {len(offers_df)} raw observations. ---")
    print(f"--- SCHEMA CHECK: {offers_df.columns.tolist()[:5]}... ---")

except Exception as e:
    print(f"!!! INGESTION FAILURE !!!")
    print(f"Error: {e}")

# Preview
offers_df.head(3)

--- ENVIRONMENT CLEAN. NO ROGUE FILES DETECTED. ---
--- PANDAS VERSION: 2.2.2 LOADED SUCCESSFULLY ---
--- AUTHENTICATING USER ---
--- INGESTING: 'raw_Offers' // 'diamond_offers' ---
--- SUCCESS: Ingested 4765 raw observations. ---
--- SCHEMA CHECK: ['offer_id', 'session_fk', 'ocr_fk', 'image_content_hash', 'offer_timestamp']... ---


,offer_id,session_fk,ocr_fk,image_content_hash,offer_timestamp,product_category_fk,upfront_fare,time_to_pickup_sec,dist_to_pickup_km,est_trip_time_sec,...,inferred_agent_lat,inferred_agent_lon,inferred_agent_bearing,inferred_agent_speed_mps,interpolation_quality_fk,is_imputed,special_note_raw,comment_1,comment_2,record_status_fk
0,OF00001,SID0001,2b6e35942cb35b11e621e758d72bd25752faaa80b68a55...,6f53fbeb55d097dbd8c1087a537cbe4589a05eaea7398d...,2025:08:22 06:44:33,uberx,204.24,840,4.90,2580,...,,,,0,UNANCHORED,FALSE,"Identidad verificada (Identity verified), +$38...",,,valid
1,OF00002,SID0001,33df0757502c3dc1c632c9a812473183ae427eb35d18af...,fe824e63f500f8c99f017cc6d975d087f9d529e7a26997...,2025:08:22 06:45:06,uberx,173.86,300,1.50,2400,...,,,,0,UNANCHORED,FALSE,"Identidad verificada (Identity verified), +$8....",,,valid
2,OF00003,SID0001,0343d4234ef85474adcff2430891b7473c274bbc9cb30f...,64f34e64aaed7113e732d1434d857ac4de639b89626cf4...,2025:08:22 06:45:28,uberx,136.53,,,,...,,,,,,TRUE,,,,invalid_non_offer


In [ ]:
# --- CELL 2: FEATURE ENGINEERING - STRATEGIC ALIGNMENT LAYER ---

import numpy as np
import pandas as pd

# --- 0. CASTING LAYER (Critical for gspread data) ---
# Convert coordinates from String to Float for math operations
numeric_cols = ['inferred_agent_lat', 'inferred_agent_lon', 'dropoff_lat', 'dropoff_lon']
for col in numeric_cols:
    offers_df[col] = pd.to_numeric(offers_df[col], errors='coerce')

print("--- DATA TYPES CAST TO NUMERIC FOR VECTOR MATH ---")

# --- 1. PROTOCOL CHER AMI: THE HOMING BEACON ---
# "Mon Ami, I am tired, but the message must get through."
# The fixed point of safety and rest amidst the chaos of the CDMX grid.
CHER_AMI_LAT = 19.4288788
CHER_AMI_LON = -99.1747491

def calculate_home_alignment(row):
    """
    Calculates the Cosine Similarity between the Agent->Sanctuary vector
    and the Agent->Dropoff vector.

    Returns:
        float: -1.0 (Flying into the fog) to +1.0 (Flying home).
        np.nan: If the signal is lost (missing coords or zero magnitude).
    """
    # 1. SIGNAL INTEGRITY CHECK
    # Using the Canonical Headers provided
    required_cols = ['inferred_agent_lat', 'inferred_agent_lon',
                     'dropoff_lat', 'dropoff_lon']

    # If any coordinate is NaN (was empty string or conversion failed), abort
    if row[required_cols].isnull().any():
        return np.nan

    # 2. DEFINE THE VECTORS (The Flight Paths)

    # The Homing Vector: From Agent TO Cher Ami (Home)
    # (Target - Current)
    vec_homing = np.array([
        CHER_AMI_LAT - row['inferred_agent_lat'],
        CHER_AMI_LON - row['inferred_agent_lon']
    ])

    # The Mission Vector: From Agent TO Offer Dropoff
    # (Target - Current)
    vec_mission = np.array([
        row['dropoff_lat'] - row['inferred_agent_lat'],
        row['dropoff_lon'] - row['inferred_agent_lon']
    ])

    # 3. CALCULATE MAGNITUDES (Distance of flight)
    norm_homing = np.linalg.norm(vec_homing)
    norm_mission = np.linalg.norm(vec_mission)

    # 4. ZERO-VECTOR PROTECTION
    # If agent is AT the sanctuary or AT the dropoff, direction is undefined.
    if norm_homing == 0 or norm_mission == 0:
        return np.nan

    # 5. CALCULATE COSINE SIMILARITY (The Alignment)
    # Formula: (Homing . Mission) / (|Homing| * |Mission|)
    dot_product = np.dot(vec_homing, vec_mission)
    alignment_score = dot_product / (norm_homing * norm_mission)

    # Clip result to [-1, 1] to handle floating point ghosts
    return np.clip(alignment_score, -1.0, 1.0)

# --- 2. EXECUTION: RELEASE THE BIRDS ---
print("--- CHER AMI: Releasing homing pigeons for all observations... ---")

# Apply the function
offers_df['home_vector_alignment_score'] = offers_df.apply(calculate_home_alignment, axis=1)

# --- 3. EXPORT THE SILVER LAYER ---
# Saving the enriched data to Drive
print(f"--- SAVING ENRICHED DATA TO: {OUTPUT_CSV_PATH} ---")
# Ensure the directory exists (just in case)
import os
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

offers_df.to_csv(OUTPUT_CSV_PATH, index=False)

# --- 4. VALIDATION REPORT ---
valid_signals = offers_df['home_vector_alignment_score'].notnull().sum()
total_missions = len(offers_df)
mean_score = offers_df['home_vector_alignment_score'].mean()

print(f"\n--- MISSION REPORT ---")
print(f"Total Flights Attempted: {total_missions}")
print(f"Messages Delivered (Valid Scores): {valid_signals}")
print(f"Average Alignment (The Pull Home): {mean_score:.4f}")
print("----------------------")

# Quick check of the new column
offers_df[['offer_id', 'inferred_agent_lat', 'dropoff_lat', 'home_vector_alignment_score']].head()

--- DATA TYPES CAST TO NUMERIC FOR VECTOR MATH ---
--- CHER AMI: Releasing homing pigeons for all observations... ---
--- SAVING ENRICHED DATA TO: /content/drive/MyDrive/_Pienza/Assets/Phase_1/cher_ami_v1.csv ---

--- MISSION REPORT ---
Total Flights Attempted: 4765
Messages Delivered (Valid Scores): 4544
Average Alignment (The Pull Home): 0.1681
----------------------


,offer_id,inferred_agent_lat,dropoff_lat,home_vector_alignment_score
0,OF00001,NaN,19.543821,NaN
1,OF00002,NaN,19.427621,NaN
2,OF00003,NaN,NaN,NaN
3,OF00004,19.444072,19.427999,0.240136
4,OF00005,19.435358,19.481590,-0.950506
